In [3]:
import pandas as pd
import numpy as np
import os

In [4]:
data_path = "../../data/"

orders_path = os.path.join(data_path, "orders.csv")
order_products_path = os.path.join(data_path, "order_products__prior.csv")

orders = pd.read_csv(orders_path)
order_products = pd.read_csv(order_products_path)

print(f"Orders shape: {orders.shape}")
print(f"Order products shape: {order_products.shape}")

Orders shape: (3421083, 7)
Order products shape: (32434489, 4)


### Merging Orders and Products  
Μerge `orders.csv` and `order_products__prior.csv` using `order_id` to connect each product purchase (`product_id`) with the user (`user_id`) who made it.  
Only the necessary columns are kept to reduce memory usage.

In [5]:
# Keep only necessary columns to reduce memory usage
merged_df = pd.merge(
    order_products[['order_id', 'product_id']],
    orders[['order_id', 'user_id']],
    on='order_id',
    how='inner'
)

print("Merged shape:", merged_df.shape)
merged_df.head()

Merged shape: (32434489, 3)


,order_id,product_id,user_id
0,2,33120,202279
1,2,28985,202279
2,2,9327,202279
3,2,45918,202279
4,2,30035,202279


### Calculating Term Frequency (TF)  
TF measures how often a user buys a specific product relative to their total purchases.  
It helps normalize purchase frequency per user, so frequent buyers are comparable to infrequent ones.

In [7]:
# TF = (times user bought product) / (total products bought by that user)
tf = merged_df.groupby(['user_id', 'product_id']).size().reset_index(name='purchase_count')
total_products_per_user = merged_df.groupby('user_id').size().reset_index(name='total_products')

tf = pd.merge(tf, total_products_per_user, on='user_id', how='left')
tf['tf'] = tf['purchase_count'] / tf['total_products']

print("TF calculated:", tf.shape)
tf.head(10)

TF calculated: (13307953, 5)


,user_id,product_id,purchase_count,total_products,tf
0,1,196,10,59,0.169492
1,1,10258,9,59,0.152542
2,1,10326,1,59,0.016949
3,1,12427,10,59,0.169492
4,1,13032,3,59,0.050847
5,1,13176,2,59,0.033898
6,1,14084,1,59,0.016949
7,1,17122,1,59,0.016949
8,1,25133,8,59,0.135593
9,1,26088,2,59,0.033898


### Calculating Inverse Document Frequency (IDF)  
IDF measures how unique or common a product is across all users.  
Products bought by many users get a lower IDF (less distinctive), while rare ones get a higher value.

In [5]:
# IDF = log(total users / users who bought this product)
total_users = orders['user_id'].nunique()
users_per_product = merged_df.groupby('product_id')['user_id'].nunique().reset_index(name='users_who_bought')

# Avoid division by zero
users_per_product['users_who_bought'] = users_per_product['users_who_bought'].replace(0, 1)
users_per_product['idf'] = np.log(total_users / users_per_product['users_who_bought'])

print("IDF calculated:", users_per_product.shape)
users_per_product.head()

IDF calculated: (49677, 3)


,product_id,users_who_bought,idf
0,1,716,5.662965
1,2,78,7.879937
2,3,74,7.932580
3,4,182,7.032639
4,5,6,10.444886


In [6]:
tf_idf = pd.merge(
    tf[['user_id', 'product_id', 'tf']],
    users_per_product[['product_id', 'idf']],
    on='product_id',
    how='left'
)

tf_idf['tfidf_score'] = tf_idf['tf'] * tf_idf['idf']

print("TF-IDF scores calculated:", tf_idf.shape)
tf_idf.head()

TF-IDF scores calculated: (13307953, 5)


,user_id,product_id,tf,idf,tfidf_score
0,1,196,0.169492,3.249449,0.550754
1,1,10258,0.152542,5.914080,0.902148
2,1,10326,0.016949,4.675004,0.079237
3,1,12427,0.169492,4.810692,0.815371
4,1,13032,0.050847,5.077354,0.258171


In [7]:
results_path = "../results/"
os.makedirs(results_path, exist_ok=True)

final_tfidf_scores = tf_idf[['user_id', 'product_id', 'tfidf_score']]

final_tfidf_scores.to_parquet(os.path.join(results_path, "user_product_tfidf_scores.parquet"), index=False)
final_tfidf_scores.to_csv(os.path.join(results_path, "user_product_tfidf_scores.csv"), index=False)

print("Saved TF-IDF results to '../results/'")
final_tfidf_scores.head()

Saved TF–IDF results to '../results/'


,user_id,product_id,tfidf_score
0,1,196,0.550754
1,1,10258,0.902148
2,1,10326,0.079237
3,1,12427,0.815371
4,1,13032,0.258171


### Interpreting TF–IDF Scores  
Higher TF–IDF values indicate products that a user consistently buys but are not common among others.  
These scores are ideal for building personalized product recommendations or user profiles.